# 13. Multi-Target DGIdb and Open Targets Cross-Check

This notebook cross-checks the multi-target therapeutic candidates against two external evidence sources:

- **DGIdb**: known drug-gene interaction evidence.
- **Open Targets**: disease associations for each target.

Goal:
- Confirm whether candidate drugs are found in external drug-gene interaction data.
- Confirm which diseases are strongly associated with each target.
- Save raw API responses and processed CSV summaries.

This notebook is still part of the data preparation stage. It does not build the RAG app yet.

In [13]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

In [14]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DGIDB_DIR = DATA_DIR / "raw" / "dgidb"
RAW_OPENTARGETS_DIR = DATA_DIR / "raw" / "opentargets"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DGIDB_DIR.mkdir(parents=True, exist_ok=True)
RAW_OPENTARGETS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RECOMMENDATIONS_FILE = PROCESSED_DIR / "multi_target_drug_recommendations.csv"
TARGETS_FILE = PROCESSED_DIR / "multi_target_chembl_targets.csv"

print("Project root:", PROJECT_ROOT)
print("DGIdb raw folder:", RAW_DGIDB_DIR)
print("Open Targets raw folder:", RAW_OPENTARGETS_DIR)
print("Processed folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
DGIdb raw folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/dgidb
Open Targets raw folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/opentargets
Processed folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


## 1. Load Project Data

The recommended drug file comes from notebook `09_multi_target_chembl_exploration.ipynb`.

In [15]:
if not RECOMMENDATIONS_FILE.exists():
    raise FileNotFoundError(
        f"Missing {RECOMMENDATIONS_FILE}. Run notebook 09_multi_target_chembl_exploration.ipynb first."
    )

recommendations_df = pd.read_csv(RECOMMENDATIONS_FILE)
print("Recommendation rows:", len(recommendations_df))
display(recommendations_df.head())

if TARGETS_FILE.exists():
    targets_df = pd.read_csv(TARGETS_FILE)
else:
    targets_df = recommendations_df[["target_symbol", "target_display_name", "target_full_name", "target_chembl_id"]].drop_duplicates()

print("Target rows:", len(targets_df))
display(targets_df)

Recommendation rows: 207


,target_symbol,target_display_name,target_full_name,target_chembl_id,target_pref_name,drug_name,molecule_chembl_id,molecule_type,action_type,mechanism_of_action,approval_status,max_phase,first_approval
0,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ALECTINIB HYDROCHLORIDE,CHEMBL3707320,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2015.0
1,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,ASP-3026,CHEMBL3545360,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
2,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,BRIGATINIB,CHEMBL3545311,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2017.0
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CEP-37440,CHEMBL3951811,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Investigational (Phase 1),1.0,NaN
4,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,CERITINIB,CHEMBL2403108,Small molecule,INHIBITOR,ALK tyrosine kinase receptor inhibitor,Approved,4.0,2014.0


Target rows: 8


,target_symbol,target_display_name,target_full_name,target_chembl_id,target_pref_name,target_organism,target_type,resolution_status
0,EGFR,EGFR,Epidermal growth factor receptor,CHEMBL203,Epidermal growth factor receptor,Homo sapiens,SINGLE PROTEIN,resolved
1,ERBB2,HER2,Receptor tyrosine-protein kinase erbB-2,CHEMBL1824,Receptor tyrosine-protein kinase erbB-2,Homo sapiens,SINGLE PROTEIN,resolved
2,BRAF,BRAF,Serine/threonine-protein kinase B-raf,CHEMBL5145,Serine/threonine-protein kinase B-raf,Homo sapiens,SINGLE PROTEIN,resolved
3,ALK,ALK,ALK tyrosine kinase receptor,CHEMBL4247,ALK tyrosine kinase receptor,Homo sapiens,SINGLE PROTEIN,resolved
4,KRAS,KRAS,GTPase KRas,CHEMBL2189121,GTPase KRas,Homo sapiens,SINGLE PROTEIN,resolved
5,VEGFA,VEGFA,Vascular endothelial growth factor A,CHEMBL1783,"Vascular endothelial growth factor A, long form",Homo sapiens,SINGLE PROTEIN,resolved
6,MET,MET,Hepatocyte growth factor receptor,CHEMBL3717,Hepatocyte growth factor receptor,Homo sapiens,SINGLE PROTEIN,resolved
7,PIK3CA,PIK3CA,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",CHEMBL4005,"Phosphatidylinositol 4,5-bisphosphate 3-kinase...",Homo sapiens,SINGLE PROTEIN,resolved


In [16]:
TARGET_ENSEMBL_IDS = {
    "EGFR": "ENSG00000146648",
    "ERBB2": "ENSG00000141736",
    "BRAF": "ENSG00000157764",
    "ALK": "ENSG00000171094",
    "KRAS": "ENSG00000133703",
    "VEGFA": "ENSG00000112715",
    "MET": "ENSG00000105976",
    "PIK3CA": "ENSG00000121879",
}

TARGET_ALIASES = {
    "EGFR": ["EGFR"],
    "ERBB2": ["ERBB2", "HER2"],
    "BRAF": ["BRAF"],
    "ALK": ["ALK"],
    "KRAS": ["KRAS"],
    "VEGFA": ["VEGFA", "VEGF"],
    "MET": ["MET"],
    "PIK3CA": ["PIK3CA"],
}

MAX_DGIDB_DRUGS_TO_DISPLAY = 20
MAX_OPENTARGETS_DISEASES = 25

print("Targets configured:", list(TARGET_ENSEMBL_IDS))

Targets configured: ['EGFR', 'ERBB2', 'BRAF', 'ALK', 'KRAS', 'VEGFA', 'MET', 'PIK3CA']


## 2. Helper Functions

In [17]:
def normalize_name(value):
    """Normalize drug names for matching across sources."""
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]+", "", str(value).upper())


def safe_join(values, limit=8):
    cleaned = [str(value).strip() for value in values if pd.notna(value) and str(value).strip()]
    unique_values = list(dict.fromkeys(cleaned))
    return " | ".join(unique_values[:limit])


def graphql_post(url, query, variables=None, retries=3, pause=2):
    """POST to a GraphQL endpoint with simple retry handling."""
    payload = {"query": query, "variables": variables or {}}
    for attempt in range(retries):
        try:
            response = requests.post(url, json=payload, timeout=(10, 90))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}; retrying")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}; retrying")
        time.sleep(pause)
    return {"errors": [{"message": "request failed after retries"}]}


def target_project_drug_norms(target_symbol):
    subset = recommendations_df[recommendations_df["target_symbol"] == target_symbol]
    return {normalize_name(value) for value in subset["drug_name"].dropna().unique()}

## 3. Query DGIdb

DGIdb provides drug-gene interaction records. This is useful for checking whether our candidate drugs appear in another interaction source.

In [18]:
DGIDB_GRAPHQL_URL = "https://dgidb.org/api/graphql"

DGIDB_QUERY = """query interactionsByGene($geneNames: [String!]!) {
  genes(names: $geneNames) {
    nodes {
      name
      conceptId
      interactions {
        interactionScore
        drug { name conceptId }
        interactionTypes { type directionality }
        sources { sourceDbName }
      }
    }
  }
}"""

dgidb_raw = {}

for target_symbol in sorted(targets_df["target_symbol"].dropna().unique()):
    gene_names = TARGET_ALIASES.get(target_symbol, [target_symbol])
    print(f"Querying DGIdb for {target_symbol}: {gene_names}")
    response = graphql_post(
        DGIDB_GRAPHQL_URL,
        DGIDB_QUERY,
        variables={"geneNames": gene_names},
    )
    dgidb_raw[target_symbol] = {
        "gene_names": gene_names,
        "response": response,
    }
    time.sleep(0.5)

raw_dgidb_file = RAW_DGIDB_DIR / "multi_target_dgidb_raw.json"
with raw_dgidb_file.open("w") as f:
    json.dump(dgidb_raw, f, indent=2)

print("Saved:", raw_dgidb_file)

Querying DGIdb for ALK: ['ALK']
Querying DGIdb for BRAF: ['BRAF']
Querying DGIdb for EGFR: ['EGFR']
Querying DGIdb for ERBB2: ['ERBB2', 'HER2']
Querying DGIdb for KRAS: ['KRAS']
Querying DGIdb for MET: ['MET']
Querying DGIdb for PIK3CA: ['PIK3CA']
Querying DGIdb for VEGFA: ['VEGFA', 'VEGF']
Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/dgidb/multi_target_dgidb_raw.json


In [19]:
dgidb_records = []

for target_symbol, payload in dgidb_raw.items():
    project_drugs = target_project_drug_norms(target_symbol)
    response = payload.get("response") or {}
    gene_nodes = (((response.get("data") or {}).get("genes") or {}).get("nodes") or [])

    for gene in gene_nodes:
        gene_name = gene.get("name")
        gene_concept_id = gene.get("conceptId")
        for interaction in gene.get("interactions", []) or []:
            drug = interaction.get("drug", {}) or {}
            interaction_types = interaction.get("interactionTypes", []) or []
            sources = interaction.get("sources", []) or []
            drug_name = drug.get("name")
            normalised_drug_name = normalize_name(drug_name)

            dgidb_records.append({
                "target_symbol": target_symbol,
                "gene_name": gene_name,
                "gene_concept_id": gene_concept_id,
                "drug_name": drug_name,
                "normalised_drug_name": normalised_drug_name,
                "drug_concept_id": drug.get("conceptId"),
                "interaction_score": interaction.get("interactionScore"),
                "interaction_types": safe_join([item.get("type") for item in interaction_types]),
                "directionality": safe_join([item.get("directionality") for item in interaction_types]),
                "sources": safe_join([item.get("sourceDbName") for item in sources], limit=12),
                "matches_project_drug": normalised_drug_name in project_drugs,
                "source": "DGIdb",
            })

dgidb_interactions_df = pd.DataFrame(dgidb_records)

expected_dgidb_columns = [
    "target_symbol", "gene_name", "gene_concept_id", "drug_name", "normalised_drug_name",
    "drug_concept_id", "interaction_score", "interaction_types", "directionality",
    "sources", "matches_project_drug", "source",
]
if dgidb_interactions_df.empty:
    dgidb_interactions_df = pd.DataFrame(columns=expected_dgidb_columns)
else:
    dgidb_interactions_df = dgidb_interactions_df[expected_dgidb_columns]

print("DGIdb interaction rows:", len(dgidb_interactions_df))
display(dgidb_interactions_df.head(30))

DGIdb interaction rows: 1431


,target_symbol,gene_name,gene_concept_id,drug_name,normalised_drug_name,drug_concept_id,interaction_score,interaction_types,directionality,sources,matches_project_drug,source
0,ALK,ALK,hgnc:427,HESPERADIN,HESPERADIN,iuphar.ligand:8354,0.009875,inhibitor,INHIBITORY,DTC,False,DGIdb
1,ALK,ALK,hgnc:427,ENSARTINIB,ENSARTINIB,rxcui:2712546,4.345163,inhibitor,INHIBITORY,PharmGKB | CancerCommons | MyCancerGenome | CK...,True,DGIdb
2,ALK,ALK,hgnc:427,BRIGATINIB,BRIGATINIB,rxcui:1921217,2.852885,inhibitor,INHIBITORY,PharmGKB | COSMIC | CGI | MyCancerGenome | Onc...,True,DGIdb
3,ALK,ALK,hgnc:427,EG5 KINESIN-RELATED MOTOR PROTEIN INHIBITOR 4S...,EG5KINESINRELATEDMOTORPROTEININHIBITOR4SC205,ncit:C90557,0.263343,,,CKB-CORE,False,DGIdb
4,ALK,ALK,hgnc:427,CRIZOTINIB,CRIZOTINIB,rxcui:1148495,2.164681,inhibitor,INHIBITORY,PharmGKB | DTC | CancerCommons | COSMIC | MyCa...,True,DGIdb
5,ALK,ALK,hgnc:427,CARBAMAZEPINE,CARBAMAZEPINE,rxcui:2002,0.016459,,,PharmGKB,False,DGIdb
6,ALK,ALK,hgnc:427,FILGOTINIB,FILGOTINIB,ncit:C155799,0.131672,,,CKB-CORE,False,DGIdb
7,ALK,ALK,hgnc:427,UNECRITINIB,UNECRITINIB,iuphar.ligand:12092,0.395015,inhibitor,INHIBITORY,GuideToPharmacology,False,DGIdb
8,ALK,ALK,hgnc:427,ALECTINIB,ALECTINIB,rxcui:1727455,4.582172,inhibitor,INHIBITORY,PharmGKB | COSMIC | CGI | MyCancerGenome | Onc...,False,DGIdb
9,ALK,ALK,hgnc:427,BELIZATINIB,BELIZATINIB,ncit:C169804,0.237009,inhibitor,INHIBITORY,MyCancerGenome | CKB-CORE | GuideToPharmacolog...,False,DGIdb


In [20]:
dgidb_file = PROCESSED_DIR / "multi_target_dgidb_interactions.csv"
dgidb_interactions_df.to_csv(dgidb_file, index=False)

print("Saved:", dgidb_file)
print("Rows:", len(dgidb_interactions_df))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_dgidb_interactions.csv
Rows: 1431


## 4. Query Open Targets

Open Targets helps us understand which diseases are associated with each target. This is useful when we later build evidence-grounded therapeutic strategy answers.

In [21]:
OPENTARGETS_GRAPHQL_URL = "https://api.platform.opentargets.org/api/v4/graphql"

OPENTARGETS_QUERY = """query targetAssociatedDiseases($ensemblId: String!, $size: Int!) {
  target(ensemblId: $ensemblId) {
    id
    approvedSymbol
    approvedName
    associatedDiseases(page: { index: 0, size: $size }) {
      count
      rows {
        score
        disease { id name }
      }
    }
  }
}"""

opentargets_raw = {}

for target_symbol in sorted(targets_df["target_symbol"].dropna().unique()):
    ensembl_id = TARGET_ENSEMBL_IDS.get(target_symbol)
    if not ensembl_id:
        print(f"Skipping {target_symbol}: no Ensembl ID configured")
        continue

    print(f"Querying Open Targets for {target_symbol}: {ensembl_id}")
    response = graphql_post(
        OPENTARGETS_GRAPHQL_URL,
        OPENTARGETS_QUERY,
        variables={"ensemblId": ensembl_id, "size": MAX_OPENTARGETS_DISEASES},
    )
    opentargets_raw[target_symbol] = {
        "ensembl_id": ensembl_id,
        "response": response,
    }
    time.sleep(0.5)

raw_opentargets_file = RAW_OPENTARGETS_DIR / "multi_target_opentargets_raw.json"
with raw_opentargets_file.open("w") as f:
    json.dump(opentargets_raw, f, indent=2)

print("Saved:", raw_opentargets_file)

Querying Open Targets for ALK: ENSG00000171094
Querying Open Targets for BRAF: ENSG00000157764
Querying Open Targets for EGFR: ENSG00000146648
Querying Open Targets for ERBB2: ENSG00000141736
Querying Open Targets for KRAS: ENSG00000133703
Querying Open Targets for MET: ENSG00000105976
Querying Open Targets for PIK3CA: ENSG00000121879
Querying Open Targets for VEGFA: ENSG00000112715
Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/opentargets/multi_target_opentargets_raw.json


In [22]:
opentargets_records = []

for target_symbol, payload in opentargets_raw.items():
    response = payload.get("response") or {}
    target_data = ((response.get("data") or {}).get("target") or {})
    associated_diseases = target_data.get("associatedDiseases") or {}
    disease_rows = associated_diseases.get("rows") or []

    for row in disease_rows:
        disease = row.get("disease") or {}
        opentargets_records.append({
            "target_symbol": target_symbol,
            "target_ensembl_id": target_data.get("id") or payload.get("ensembl_id"),
            "approved_symbol": target_data.get("approvedSymbol"),
            "target_full_name": target_data.get("approvedName"),
            "disease_id": disease.get("id"),
            "disease_name": disease.get("name"),
            "association_score": row.get("score"),
            "total_associated_disease_count": associated_diseases.get("count"),
            "source": "Open Targets",
        })

opentargets_associations_df = pd.DataFrame(opentargets_records)

expected_ot_columns = [
    "target_symbol", "target_ensembl_id", "approved_symbol", "target_full_name",
    "disease_id", "disease_name", "association_score", "total_associated_disease_count", "source",
]
if opentargets_associations_df.empty:
    opentargets_associations_df = pd.DataFrame(columns=expected_ot_columns)
else:
    opentargets_associations_df = opentargets_associations_df[expected_ot_columns]

print("Open Targets disease association rows:", len(opentargets_associations_df))
display(opentargets_associations_df.head(30))

Open Targets disease association rows: 200


,target_symbol,target_ensembl_id,approved_symbol,target_full_name,disease_id,disease_name,association_score,total_associated_disease_count,source
0,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0005072,neuroblastoma,0.837456,1443,Open Targets
1,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0013083,"neuroblastoma, susceptibility to, 3",0.760482,1443,Open Targets
2,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0005233,non-small cell lung carcinoma,0.757065,1443,Open Targets
3,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0004992,cancer,0.725284,1443,Open Targets
4,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0005070,neoplasm,0.661377,1443,Open Targets
5,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0008903,lung cancer,0.538290,1443,Open Targets
6,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0020325,anaplastic large cell lymphoma,0.514620,1443,Open Targets
7,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0005061,lung adenocarcinoma,0.488159,1443,Open Targets
8,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0005097,squamous cell lung carcinoma,0.473959,1443,Open Targets
9,ALK,ENSG00000171094,ALK,ALK receptor tyrosine kinase,MONDO_0005062,lymphoma,0.465545,1443,Open Targets


In [23]:
opentargets_file = PROCESSED_DIR / "multi_target_opentargets_associations.csv"
opentargets_associations_df.to_csv(opentargets_file, index=False)

print("Saved:", opentargets_file)
print("Rows:", len(opentargets_associations_df))

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_opentargets_associations.csv
Rows: 200


## 5. Build Cross-Check Summary

This summary tells us whether each target has external interaction evidence and disease-association evidence.

In [24]:
base_targets_df = targets_df[["target_symbol"]].drop_duplicates().copy()
if "target_display_name" in targets_df.columns:
    base_targets_df = base_targets_df.merge(
        targets_df[["target_symbol", "target_display_name"]].drop_duplicates(),
        on="target_symbol",
        how="left",
    )
else:
    base_targets_df["target_display_name"] = base_targets_df["target_symbol"]

recommendation_counts_df = recommendations_df.groupby("target_symbol").agg(
    project_candidate_drug_count=("drug_name", "nunique"),
).reset_index()

if dgidb_interactions_df.empty:
    dgidb_summary_df = pd.DataFrame(columns=[
        "target_symbol", "dgidb_interaction_count", "dgidb_unique_drug_count", "dgidb_project_drug_match_count",
        "top_dgidb_project_matches",
    ])
else:
    matched_only = dgidb_interactions_df[dgidb_interactions_df["matches_project_drug"] == True]
    dgidb_summary_df = dgidb_interactions_df.groupby("target_symbol").agg(
        dgidb_interaction_count=("drug_name", "size"),
        dgidb_unique_drug_count=("drug_name", "nunique"),
    ).reset_index()
    match_summary_df = matched_only.groupby("target_symbol").agg(
        dgidb_project_drug_match_count=("drug_name", "nunique"),
        top_dgidb_project_matches=("drug_name", safe_join),
    ).reset_index()
    dgidb_summary_df = dgidb_summary_df.merge(match_summary_df, on="target_symbol", how="left")

if opentargets_associations_df.empty:
    opentargets_summary_df = pd.DataFrame(columns=[
        "target_symbol", "opentargets_disease_rows", "opentargets_total_disease_count", "top_opentargets_diseases",
    ])
else:
    opentargets_summary_df = opentargets_associations_df.sort_values(
        ["target_symbol", "association_score"], ascending=[True, False]
    ).groupby("target_symbol").agg(
        opentargets_disease_rows=("disease_name", "size"),
        opentargets_total_disease_count=("total_associated_disease_count", "max"),
        top_opentargets_diseases=("disease_name", safe_join),
    ).reset_index()

crosscheck_summary_df = (
    base_targets_df
    .merge(recommendation_counts_df, on="target_symbol", how="left")
    .merge(dgidb_summary_df, on="target_symbol", how="left")
    .merge(opentargets_summary_df, on="target_symbol", how="left")
)

count_columns = [
    "project_candidate_drug_count", "dgidb_interaction_count", "dgidb_unique_drug_count",
    "dgidb_project_drug_match_count", "opentargets_disease_rows", "opentargets_total_disease_count",
]
for column in count_columns:
    if column in crosscheck_summary_df.columns:
        crosscheck_summary_df[column] = crosscheck_summary_df[column].fillna(0).astype(int)

for column in ["top_dgidb_project_matches", "top_opentargets_diseases"]:
    if column in crosscheck_summary_df.columns:
        crosscheck_summary_df[column] = crosscheck_summary_df[column].fillna("")

crosscheck_summary_df["has_dgidb_evidence"] = crosscheck_summary_df["dgidb_interaction_count"] > 0
crosscheck_summary_df["has_project_drug_match_in_dgidb"] = crosscheck_summary_df["dgidb_project_drug_match_count"] > 0
crosscheck_summary_df["has_opentargets_disease_evidence"] = crosscheck_summary_df["opentargets_disease_rows"] > 0
crosscheck_summary_df["source_status"] = crosscheck_summary_df.apply(
    lambda row: "working" if row["has_dgidb_evidence"] or row["has_opentargets_disease_evidence"] else "no evidence found",
    axis=1,
)

display(crosscheck_summary_df)

,target_symbol,target_display_name,project_candidate_drug_count,dgidb_interaction_count,dgidb_unique_drug_count,dgidb_project_drug_match_count,top_dgidb_project_matches,opentargets_disease_rows,opentargets_total_disease_count,top_opentargets_diseases,has_dgidb_evidence,has_project_drug_match_in_dgidb,has_opentargets_disease_evidence,source_status
0,EGFR,EGFR,76,291,262,72,AUMOLERTINIB | ABIVERTINIB | FUTUXIMAB | RINDO...,25,6459,non-small cell lung carcinoma | lung adenocarc...,True,True,True,working
1,ERBB2,HER2,40,193,174,39,TAK-285 | PERTUZUMAB | SAPITINIB | AFATINIB DI...,25,1910,non-small cell lung carcinoma | cancer | gastr...,True,True,True,working
2,BRAF,BRAF,14,220,210,14,LIFIRAFENIB | REGORAFENIB | ENCORAFENIB | PLIX...,25,3139,cardiofaciocutaneous syndrome | Noonan syndrom...,True,True,True,working
3,ALK,ALK,11,134,126,11,ENSARTINIB | BRIGATINIB | CRIZOTINIB | CEP-374...,25,1443,"neuroblastoma | neuroblastoma, susceptibility ...",True,True,True,working
4,KRAS,KRAS,2,122,117,2,SOTORASIB | ADAGRASIB,25,2404,Noonan syndrome | Noonan syndrome 3 | cardiofa...,True,True,True,working
5,VEGFA,VEGFA,13,56,54,12,RANIBIZUMAB | BEVACIZUMAB 111IN | AFLIBERCEPT ...,25,6064,diabetic retinopathy | non-small cell lung car...,True,True,True,working
6,MET,MET,39,133,118,37,CABOZANTINIB S-MALATE | GLESATINIB | JNJ-38877...,25,2315,papillary renal cell carcinoma | hereditary pa...,True,True,True,working
7,PIK3CA,PIK3CA,9,282,262,9,INAVOLISIB | COPANLISIB | COPANLISIB HYDROCHLO...,25,1791,megalencephaly-capillary malformation-polymicr...,True,True,True,working


In [25]:
crosscheck_file = PROCESSED_DIR / "multi_target_external_crosscheck_summary.csv"
coverage_file = PROCESSED_DIR / "multi_target_dgidb_opentargets_coverage_summary.csv"

crosscheck_summary_df.to_csv(crosscheck_file, index=False)
crosscheck_summary_df.to_csv(coverage_file, index=False)

print("Saved:", crosscheck_file)
print("Saved:", coverage_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_external_crosscheck_summary.csv
Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_dgidb_opentargets_coverage_summary.csv


## 6. Final Check

After running this notebook, confirm that all targets have a row in the coverage summary.

In [26]:
print("DGIdb and Open Targets Multi-Target Cross-Check Complete")
print("=" * 70)
print("Targets:", crosscheck_summary_df["target_symbol"].nunique())
print("DGIdb interaction rows:", len(dgidb_interactions_df))
print("Open Targets association rows:", len(opentargets_associations_df))
print("Files created:")
print("-", raw_dgidb_file)
print("-", raw_opentargets_file)
print("-", dgidb_file)
print("-", opentargets_file)
print("-", crosscheck_file)
print("-", coverage_file)

display(crosscheck_summary_df)

DGIdb and Open Targets Multi-Target Cross-Check Complete
Targets: 8
DGIdb interaction rows: 1431
Open Targets association rows: 200
Files created:
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/dgidb/multi_target_dgidb_raw.json
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/raw/opentargets/multi_target_opentargets_raw.json
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_dgidb_interactions.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/multi_target_opentargets_associations.csv
- /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/pr

,target_symbol,target_display_name,project_candidate_drug_count,dgidb_interaction_count,dgidb_unique_drug_count,dgidb_project_drug_match_count,top_dgidb_project_matches,opentargets_disease_rows,opentargets_total_disease_count,top_opentargets_diseases,has_dgidb_evidence,has_project_drug_match_in_dgidb,has_opentargets_disease_evidence,source_status
0,EGFR,EGFR,76,291,262,72,AUMOLERTINIB | ABIVERTINIB | FUTUXIMAB | RINDO...,25,6459,non-small cell lung carcinoma | lung adenocarc...,True,True,True,working
1,ERBB2,HER2,40,193,174,39,TAK-285 | PERTUZUMAB | SAPITINIB | AFATINIB DI...,25,1910,non-small cell lung carcinoma | cancer | gastr...,True,True,True,working
2,BRAF,BRAF,14,220,210,14,LIFIRAFENIB | REGORAFENIB | ENCORAFENIB | PLIX...,25,3139,cardiofaciocutaneous syndrome | Noonan syndrom...,True,True,True,working
3,ALK,ALK,11,134,126,11,ENSARTINIB | BRIGATINIB | CRIZOTINIB | CEP-374...,25,1443,"neuroblastoma | neuroblastoma, susceptibility ...",True,True,True,working
4,KRAS,KRAS,2,122,117,2,SOTORASIB | ADAGRASIB,25,2404,Noonan syndrome | Noonan syndrome 3 | cardiofa...,True,True,True,working
5,VEGFA,VEGFA,13,56,54,12,RANIBIZUMAB | BEVACIZUMAB 111IN | AFLIBERCEPT ...,25,6064,diabetic retinopathy | non-small cell lung car...,True,True,True,working
6,MET,MET,39,133,118,37,CABOZANTINIB S-MALATE | GLESATINIB | JNJ-38877...,25,2315,papillary renal cell carcinoma | hereditary pa...,True,True,True,working
7,PIK3CA,PIK3CA,9,282,262,9,INAVOLISIB | COPANLISIB | COPANLISIB HYDROCHLO...,25,1791,megalencephaly-capillary malformation-polymicr...,True,True,True,working
